In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 10.1 MB/s eta 0:00:00


In [3]:
from pathlib import Path
from time import perf_counter
from PIL import Image, ImageDraw
from matplotlib.patches import Rectangle

import importlib.util
import inspect
import re
import subprocess
import sys
import urllib.request
import zipfile


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
)
from ultralytics import SAM

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
DEVICE = "cuda"
PROJECT = Path("/content/drive/MyDrive/Vision/vision_unit_04")

OUTPUTS = PROJECT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUTPUTS / "evaluation_manifest.csv"

DATA_PARENT = Path("/content/datasets")
DATA = DATA_PARENT / "coco128-seg"

print("Device:", torch.cuda.get_device_name(0))
print("Manifest:", MANIFEST_PATH)

Device: Tesla T4
Manifest: /content/drive/MyDrive/Vision/vision_unit_04/outputs/evaluation_manifest.csv


**Dataset, annotations and calibration images**

In [6]:
DATA_PARENT.mkdir(parents=True, exist_ok=True)

if not (DATA / "images/train2017").exists():
    archive_path = DATA_PARENT / "coco128-seg.zip"

    urllib.request.urlretrieve(
        "https://github.com/ultralytics/assets/"
        "releases/download/v0.0.0/"
        "coco128-seg.zip",
        archive_path
    )

    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_PARENT)

In [7]:
TARGET_CLASSES = {
    0: "person",
    2: "car",
    16: "dog",
    41: "cup",
    56: "chair",
}

name_to_id = {
    class_name: class_id
    for class_id, class_name
    in TARGET_CLASSES.items()
}

manifest = pd.read_csv(MANIFEST_PATH)
manifest.head()

,image_name,split,selected_classes,selected_instances
0,000000000042.jpg,calibration,dog,1
1,000000000064.jpg,calibration,car,1
2,000000000071.jpg,calibration,car,13
3,000000000074.jpg,calibration,person|dog,7
4,000000000077.jpg,calibration,person,5


In [8]:
cal_names = manifest.loc[
    manifest["split"] == "calibration", "image_name"
].tolist()


images = {
    name: Image.open(
        DATA / "images/train2017" / name
    ).convert("RGB")
    for name in cal_names
}

In [9]:
annotations = {}

for name in cal_names:
    label_path = (
        DATA / "labels/train2017" / f"{Path(name).stem}.txt"
    )
    
    items = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            
            values = line.split()
            class_id = int(values[0])
            
            polygon_norm = np.asarray(
                values[1:], dtype=np.float32
            ).reshape(-1, 2)
            
            items.append({
                "class_id": class_id,
                "polygon_norm": polygon_norm,
            })

    annotations[name] = items
    

print("Calibration images:", len(cal_names))
print(
    "Selected-class instances:",
    sum(
        item["class_id"] in TARGET_CLASSES
        for items in annotations.values()
        for item in items
    ),
)

Calibration images: 20
Selected-class instances: 110


**Grounding DINO and SAM2.1 load**

In [10]:
LOCKED = {
    "prompt": "chair. cup. dog. car. person.",
    "box_threshold": 0.35,
    "text_threshold": 0.30,
}

GROUNDING_MODEL_ID = (
    "IDEA-Research/grounding-dino-tiny"
)

processor = AutoProcessor.from_pretrained(GROUNDING_MODEL_ID)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [11]:
model = AutoModelForZeroShotObjectDetection.from_pretrained(
        GROUNDING_MODEL_ID,
        disable_custom_kernels=True
).float().to(DEVICE).eval()

postprocess = processor.post_process_grounded_object_detection
postprocess_parameters = inspect.signature(postprocess).parameters

BOX_ARGUMENT = (
    "threshold"
    if "threshold" in postprocess_parameters
    else "box_threshold"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  689MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

In [ ]:
sam_model = SAM("sam2.1_s.pt")

def normalize_label(label):
    label = str(label).lower().strip(" .")
    return re.sub(r"^(a|an|the)\s+", "", label)

In [13]:
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    intersection_width = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    intersection_height = max(0.0, min(ay2, by2) - max(ay1, by1))

    intersection = (intersection_width * intersection_height)

    area_a = (
        max(0.0, ax2 - ax1)
        * max(0.0, ay2 - ay1)
    )

    area_b = (
        max(0.0, bx2 - bx1)
        * max(0.0, by2 - by1)
    )

    union = area_a + area_b - intersection

    return (
        intersection / union
        if union > 0
        else 0.0
    )


print("Locked configuration:", LOCKED)

Locked configuration: {'prompt': 'chair. cup. dog. car. person.', 'box_threshold': 0.35, 'text_threshold': 0.3}
